#  Theoretical Foundation 

# Part A – Theoretical Foundation

## 1. What is a Statistical Distribution?

A **statistical distribution** describes how the values of a dataset are spread and the probability of each value occurring.

---

## 2. What is a Q-Q Plot and Why is it Used?

A **Q-Q (Quantile-Quantile) Plot** is used to compare a dataset with a theoretical distribution (usually Normal) to check whether the data follows that distribution.

---

## 3. Difference Between Discrete and Continuous Distributions

| Feature  | Discrete Distribution  | Continuous Distribution |
| -------- | ---------------------- | ----------------------- |
| Values   | Countable integers     | Infinite decimal values |
| Nature   | Counting               | Measuring               |
| Function | PMF                    | PDF                     |
| Example  | Number of transactions | Transaction amount      |

---

## 4. What is a Bernoulli Distribution?

A **Bernoulli Distribution** models a single experiment with only two possible outcomes: **Success (1)** or **Failure (0)**.

---

## 5. What is a Binomial Distribution?

A **Binomial Distribution** models the number of successes obtained from a fixed number of independent Bernoulli trials.

---

## 6. Explain Log-Normal Distribution.

A **Log-Normal Distribution** describes positive, right-skewed data whose logarithm follows a Normal Distribution.

---

## 7. Explain Power Law Distribution.

A **Power Law Distribution** represents data where many values are small and a few values are extremely large.

---

## 8. What is a Box-Cox Transform?

The **Box-Cox Transform** is a mathematical technique used to reduce skewness and make data closer to a Normal Distribution.

---

## 9. Explain Poisson Distribution with an Example.

A **Poisson Distribution** models the number of events occurring within a fixed time or space interval, such as daily transactions.

---

## 10. What is Z-score Probability?

A **Z-score** measures how many standard deviations a data point is away from the mean and is used to calculate probabilities and detect outliers.

---

## 11. Differentiate Between PDF and CDF.

| Feature  | PDF (Probability Density Function)       | CDF (Cumulative Distribution Function)         |
| -------- | ---------------------------------------- | ---------------------------------------------- |
| Meaning  | Shows the probability density at a value | Shows the cumulative probability up to a value |
| Used For | Continuous distributions                 | Any distribution                               |
| Range    | Density values                           | 0 to 1                                         |
| Total    | Area under the curve equals 1            | Ends at 1                                      |


In [ ]:

import pandas as pd
df=pd.read_excel(r"C:\Users\ansh\OneDrive\Desktop\Ansh\rw\rw\MAS\pr3\spread_locator_dataset.xlsx")
df.head()



ModuleNotFoundError: No module named 'numpy'

In [ ]:
import numpy as np
from scipy.stats import bernoulli, binom

import matplotlib.pyplot as plt

transaction_occurrence = (data["transaction_status"].str.lower() == "success").astype(int)
p_bernoulli = transaction_occurrence.mean()
bernoulli_fit = bernoulli(p=p_bernoulli)

weekly_counts = (
    pd.to_datetime(data["transaction_date"])
    .dt.to_period("W")
    .value_counts()
    .sort_index()
)

n_binom = int(weekly_counts.max())
p_binom = weekly_counts.mean() / n_binom if n_binom > 0 else 0
binomial_fit = binom(n=n_binom, p=p_binom)

print(f"Bernoulli p: {p_bernoulli:.4f}")
print(f"Binomial n: {n_binom}, p: {p_binom:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bernoulli
axes[0].bar([0, 1], [1 - p_bernoulli, p_bernoulli], alpha=0.7, label="Fitted Bernoulli")
axes[0].set_xticks([0, 1])
axes[0].set_title("Bernoulli Fit: Transaction Occurrence")
axes[0].set_xlabel("Occurrence")
axes[0].set_ylabel("Probability")
axes[0].legend()

# Binomial
x = np.arange(0, n_binom + 1)
axes[1].bar(x, binomial_fit.pmf(x), alpha=0.7, label="Fitted Binomial")
axes[1].set_title("Binomial Fit: Weekly Transaction Count")
axes[1].set_xlabel("Weekly Count")
axes[1].set_ylabel("Probability")
axes[1].legend()

plt.tight_layout()
plt.show()

NameError: name 'df' is not defined

In [ ]:
#2.Fit the data to Poisson distribution (number of transactions per day).
from matplotlib.pylab import poisson
from scipy.stats import poisson

daily_counts = (pd.to_datetime(data["transaction_date"]).dt.date.value_counts().sort_index())

lambda_poisson = daily_counts.mean()
poisson_fit = poisson(mu=lambda_poisson)

print(f"Poisson lambda: {lambda_poisson:.4f}")

In [ ]:
#3. Model transaction amounts using Log-Normal and Power Law distributions.

from scipy.stats import lognorm, pareto
model_data = data["transaction_amount"].dropna()
log_normal_fit = np.log(model_data).mean(), np.log(model_data).std()
print(f"Log-Normal fit: mean={log_normal_fit[0]:.4f}, std={log_normal_fit[1]:.4f}")

power_law_fit = (model_data.min(), model_data.max())
shape_ln, loc_ln, scale_ln = lognorm.fit(model_data, floc=0)
power_shape, power_loc, power_scale = pareto.fit(model_data, floc=0)

log_normal_fit = (np.log(scale_ln), shape_ln)
power_law_fit = (power_shape, power_scale)
print(f"Power Law fit: min={power_law_fit[0]:.4f}, max={power_law_fit[1]:.4f}")
x = np.linspace(model_data.min(), model_data.max(), 500)
plt.figure(figsize=(10, 5))
plt.hist(model_data, bins=30, density=True, alpha=0.4, label="Observed")
plt.plot(x, lognorm.pdf(x, shape_ln, loc_ln, scale_ln), label="Log-Normal")
plt.plot(x, pareto.pdf(x, power_shape, power_loc, power_scale), label="Power Law")
plt.title("Transaction Amount Distribution Fits")
plt.xlabel("Transaction Amount")
plt.ylabel("Density")
plt.legend()
plt.show()

In [ ]:
import scipy.stats as stats
import matplotlib.pyplot as plt

qq_data = model_data.dropna()

fig, ax = plt.subplots(figsize=(6, 6))
stats.probplot(qq_data, dist="norm", plot=ax)
ax.set_title("Q-Q Plot of Transaction Amounts")
ax.set_xlabel("Theoretical Quantiles")
ax.set_ylabel("Sample Quantiles")
plt.tight_layout()
stats.probplot(qq_data, dist="norm")
plt.show()

In [ ]:
 # Apply Box-Cox transform to the transaction amounts
bc_data = model_data.dropna().astype(float)

shift = 0
if (bc_data <= 0).any():
    shift = abs(bc_data.min()) + 1
    bc_data = bc_data + shift

boxcox_transformed, boxcox_lambda = stats.boxcox(bc_data)

print(f"Box-Cox lambda: {boxcox_lambda:.4f}")
print(f"Applied shift: {shift:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(model_data.dropna(), bins=30, density=True, alpha=0.7)
axes[0].set_title("Original Transaction Amounts")
axes[0].set_xlabel("Transaction Amount")
axes[0].set_ylabel("Density")

axes[1].hist(boxcox_transformed, bins=30, density=True, alpha=0.7, color="orange")
axes[1].set_title("Box-Cox Transformed Amounts")
axes[1].set_xlabel("Transformed Value")
axes[1].set_ylabel("Density")

plt.tight_layout()
plt.show()

In [ ]:
# Z-scores and probability of transactions exceeding ₹5000

mu = model_data.mean()
sigma = model_data.std(ddof=0)  # population std
z_scores = (model_data - mu) / sigma

z_5000 = (5000 - mu) / sigma
prob_gt_5000_norm = stats.norm.sf(z_5000)            # Normal-approx probability
prob_gt_5000_empirical = (model_data > 5000).mean()  # Empirical probability

print(f"Mean={mu:.2f}, Std={sigma:.2f}")
print(f"Z-score for 5000: {z_5000:.4f}")
print(f"P(X>5000) (Normal approx): {prob_gt_5000_norm:.4f}")
print(f"P(X>5000) (Empirical): {prob_gt_5000_empirical:.4f}")

In [ ]:
# PDF and CDF for transaction amounts using the fitted Log-Normal model

x_vals = np.linspace(model_data.min(), model_data.max(), 500)
pdf_vals = stats.lognorm.pdf(x_vals, shape_ln, loc=loc_ln, scale=scale_ln)
cdf_vals = stats.lognorm.cdf(x_vals, shape_ln, loc=loc_ln, scale=scale_ln)

sorted_data = np.sort(model_data.values)
ecdf = np.arange(1, len(sorted_data) + 1) / len(sorted_data)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PDF
axes[0].hist(model_data, bins=30, density=True, alpha=0.35, label="Observed density")
axes[0].plot(x_vals, pdf_vals, color="crimson", linewidth=2, label="Fitted Log-Normal PDF")
axes[0].set_title("PDF of Transaction Amounts")
axes[0].set_xlabel("Transaction Amount")
axes[0].set_ylabel("Density")
axes[0].legend()

# CDF
axes[1].plot(x_vals, cdf_vals, color="navy", linewidth=2, label="Fitted Log-Normal CDF")
axes[1].step(sorted_data, ecdf, where="post", color="gray", alpha=0.7, label="Empirical CDF")
axes[1].set_title("CDF of Transaction Amounts")
axes[1].set_xlabel("Transaction Amount")
axes[1].set_ylabel("Cumulative Probability")
axes[1].legend()

plt.tight_layout()
plt.show()

mode_x = x_vals[np.argmax(pdf_vals)]
median_x = np.median(model_data)

print(f"Mode: ₹{mode_x:.2f}, Median: ₹{median_x:.2f}")